# Exercise 3 — Validate the sales

**Worked solution** · [All exercises](../index.html) · [Setup](../README.md)

**Core: about 5 minutes.** The same baseline for everyone. [Optional zoom-in](#zoom): about 6 extra minutes; choose it here if the topic interests you.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../RECOVERY.md).

In [1]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=True)
product_key, clean_products = workspace.load('product_key', 'clean_products')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 19:23:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="exercise-3"></a>
## Your task

**Which rows belong in the report, and how will we explain the rejects?**

Core budget: about 5 minutes.

Our contract rejects missing/empty product keys, invalid or missing amounts, and invalid or missing timestamps. Keep the raw values and a `reject_reason`. A valid key absent from the lookup is still an accepted sale.

Parse amounts as `DECIMAL(12, 2)` and timestamps with `yyyy-MM-dd HH:mm:ss`. Parsing failures should become null, so we can explain them. Do not silently turn missing amounts into zero.

### Predict before running

Which sale IDs will fail this contract? Does a missing product lookup make a sale invalid?

Your prediction: …

### Supplied — parse and mark bad rows

This parsing function is supplied for the core. Read its rules before using it; the optional section lets you build the parsing expressions yourself.

In [2]:
def clean_sales(raw: DataFrame) -> DataFrame:
    """Parse the exercise's UTC timestamps and amounts; retain bad input."""
    return (
        raw.withColumn("product_id", product_key(F.col("product_id")))
        .withColumn("amount", F.expr("try_cast(amount_raw AS DECIMAL(12, 2))"))
        .withColumn(
            "sold_at",
            F.try_to_timestamp("sold_at_raw", F.lit("yyyy-MM-dd HH:mm:ss")),
        )
        .withColumn(
            "reject_reason",
            F.when(
                F.col("product_id").isNull() | (F.col("product_id") == ""),
                F.lit("missing product key"),
            )
            .when(F.col("amount").isNull(), F.lit("invalid or missing amount"))
            .when(F.col("sold_at").isNull(), F.lit("invalid or missing timestamp")),
        )
    )

In [3]:
cleaned = clean_sales(raw)
cleaned.select("sale_id", "product_id", "amount", "sold_at", "reject_reason").orderBy(
    "sale_id"
).show(truncate=False)

+-------+----------+------+-------------------+----------------------------+
|sale_id|product_id|amount|sold_at            |reject_reason               |
+-------+----------+------+-------------------+----------------------------+
|s1     |B1        |25.00 |2026-09-01 09:00:00|NULL                        |
|s2     |G1        |40.00 |2026-09-01 09:05:00|NULL                        |
|s3     |B1        |15.00 |2026-09-02 10:00:00|NULL                        |
|s4     |M1        |10.00 |2026-09-02 11:00:00|NULL                        |
|s5     |B1        |10.00 |2026-09-02 12:00:00|NULL                        |
|s6     |B1        |NULL  |2026-09-02 12:05:00|invalid or missing amount   |
|s7     |G1        |NULL  |2026-09-02 12:10:00|invalid or missing amount   |
|s8     |B1        |5.00  |NULL               |invalid or missing timestamp|
+-------+----------+------+-------------------+----------------------------+



### Your code — separate accepted and rejected rows

Keep the filtering functions reusable: the stream will call them too.

In [4]:
def accepted_sales(cleaned: DataFrame) -> DataFrame:
    """Keep parsed sales with no rejection reason, including keys absent from the lookup."""
    return cleaned.filter(F.col("reject_reason").isNull())


def rejected_sales(cleaned: DataFrame) -> DataFrame:
    """Keep invalid sales and their original values for diagnosis."""
    return cleaned.filter(F.col("reject_reason").isNotNull())

In [5]:
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
rejected.select("sale_id", "amount_raw", "sold_at_raw", "reject_reason").orderBy("sale_id").show(
    truncate=False
)

+-------+----------+-------------------+----------------------------+
|sale_id|amount_raw|sold_at_raw        |reject_reason               |
+-------+----------+-------------------+----------------------------+
|s6     |oops      |2026-09-02 12:05:00|invalid or missing amount   |
|s7     |NULL      |2026-09-02 12:10:00|invalid or missing amount   |
|s8     |5.00      |not-a-date         |invalid or missing timestamp|
+-------+----------+-------------------+----------------------------+



### Check

Reconcile all eight input rows. Check which rows survive, not just how many.

In [6]:
check.validation(raw, accepted, rejected)

Validation passed: 8 input = 5 accepted + 3 rejected; M1 is accepted.


<details>
<summary>Need a nudge? Hint 1</summary>

Use the tolerant parsing functions named in the task. Test parsed values for null.

</details>

<details>
<summary>A little more help: Hint 2</summary>

A chained `when` without an `otherwise` produces null when none of its conditions matches. Filter that column with `isNull()` or `isNotNull()`.

</details>

If you need to catch up during class, use the explicit [recovery step](../RECOVERY.md#exercise-3). [Worked solution](03-validate.ipynb) — open it separately when you are ready to compare.

## Core complete

For the 60-minute lab, [skip to Save and finish](#finish). To explore this topic further, continue with the optional section below. Later core exercises do not need any of its variables.

<a id="zoom"></a>
## Optional zoom-in · about 6 minutes

These investigations make up the extra depth in a 90-minute session. Choose them independently; keep your working pipeline unchanged.

### Parseable does not always mean valid

Our contract is intentionally small. It does not, for example, reject a negative amount: refunds might be legitimate. A real pipeline needs an explicit business decision about that.

Keep `amount_raw` and `sold_at_raw` alongside parsed columns so a reject can be explained. Spark null checks use `isNull()`/`isNotNull()`, not Python's `is None`. The [parsing experiment](deeper/schemas-and-parsing.ipynb) adds malformed and missing keys/timestamps without changing the main fixture.

### Your code — build the parsing expressions

Create a separate `parsed_preview` with `sale_id`, parsed `amount` as `DECIMAL(12, 2)`, and parsed `sold_at` using `yyyy-MM-dd HH:mm:ss`. Use tolerant parsing so invalid input becomes null. Compare it with the supplied cleaner without changing that function.

In [7]:
parsed_preview = raw.select(
    "sale_id",
    F.expr("try_cast(amount_raw AS DECIMAL(12, 2))").alias("amount"),
    F.try_to_timestamp("sold_at_raw", F.lit("yyyy-MM-dd HH:mm:ss")).alias("sold_at"),
)
parsed_preview.orderBy("sale_id").show()

+-------+------+-------------------+
|sale_id|amount|            sold_at|
+-------+------+-------------------+
|     s1| 25.00|2026-09-01 09:00:00|
|     s2| 40.00|2026-09-01 09:05:00|
|     s3| 15.00|2026-09-02 10:00:00|
|     s4| 10.00|2026-09-02 11:00:00|
|     s5| 10.00|2026-09-02 12:00:00|
|     s6|  NULL|2026-09-02 12:05:00|
|     s7|  NULL|2026-09-02 12:10:00|
|     s8|  5.00|               NULL|
+-------+------+-------------------+



In [8]:
check.same_rows(cleaned.select("sale_id", "amount", "sold_at"), parsed_preview)

CSV and Parquet values agree.


<details><summary>Hint for the parsing expressions</summary>

Use `F.expr` with SQL `try_cast` for the decimal and `F.try_to_timestamp` with `F.lit` for the timestamp pattern. Both expressions should preserve a row even when parsing fails.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [9]:
workspace.save(clean_sales, accepted_sales, rejected_sales)
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Saved your functions to learner_work/solutions/answers.py


Session stopped; exercise files are under runs/run-854160eec0


Next: [Exercise 4 — Join and aggregate](04-join-aggregate.ipynb).

Want more on this topic? You can open these now, using the same saved work: [Schemas and parsing](deeper/schemas-and-parsing.ipynb).